# Exploratory Data Analysis (EDA)

We want to explore the raw data that we claimed from `EngSaf` to answer these questions:
1. It is clean enough to be used in our model training and evaluation?
2. How many missing values there?
3. How many duplications?
4. What is the semantic similarity between the train, validation and unseen data?
5. Is there data leakages between train, validation and test data?
6. What actions should be taken according to the missing, duplicated and leakage values?
7. What is the final (clean) training size will be?
8. How should we split the validation and test sets (size for each)?
9. What is the maximum length (word-based) in both train and unseen sets?
10. Is the static prompt length + maximum length is larger than the maximum model length?

In [2]:
import os
import json
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

#### The raw data that we claimed from `EngSaf` are downloaded and saved in `raw` dir

In [3]:
raw_datasets  = "raw"

dataset_files = [file for file in os.listdir(raw_datasets) if file.endswith(".csv")]

dataset_files

['val.csv', 'unseen_answers.csv', 'unseen_question.csv', 'train.csv']

## Train Dataset
Here is a summary of the insights and finds of the raw train set:
1. the data contains missing values in both `Question_id` and `Student Answer`.
2. we will drop the `Question_id` there is no need for it.
3. we will drop the rows that contains missing data because they are few (12 missing).
4. according to the column names, we will rename all the column names:
    1. Question       -> `question`
    2. Student Answer -> `student_answer`
    3. Correct Answer -> `reference_answer`
    4. output_label   -> `score`
    5. feedback       -> `rationale`
5. we missed the `mark_scheme` column, but the dataset official documentation includes it, so we'll add it during feature engineering.
6. there is a few full duplications, these we will drop them.
7. the total raw train size 3662 records.
8. there is 106 unique questions.
9. there is 3516 unique student answer.
10. there is 3614 unique feedback.
11. The distribution of `output_label` values is almost balanced, with the largest bias being label `2` and the lowest is `0`.
12. The lengths of students answers vary, from short to long, same thing for feedback which is a good thing for variation.
13. we will drop the full duplicated records.

In [13]:
train_path = os.path.join(raw_datasets, "train.csv")

train_df   = pd.read_csv(train_path)

train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3662 entries, 0 to 3661
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Question_id     2953 non-null   float64
 1   Question        3662 non-null   object 
 2   Student Answer  3650 non-null   object 
 3   Correct Answer  3662 non-null   object 
 4   output_label    3662 non-null   int64  
 5   feedback        3662 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 171.8+ KB


In [14]:
train_df.sample(5)

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
2491,30144.0,What is an orthotropic material ?,material having three perpendicular planes of ...,Orthotropic materials have 9 elastic constants...,2,Your answer captures the essence of orthotropi...
3102,76520.0,Usable life of a reservoir is,when the reservoir can still serve after it's ...,"It is the period of time, extending beyond the...",1,Your answer captures the idea that a reservo...
3207,NaN,Q1. State TRUE or FALSE and justify. No correc...,"False, True, done using iret.","False, some other process can also be schedule...",0,"""False, True"" is not a correct response format..."
3293,42541.0,Q1. State TRUE or FALSE and justify. [6 marks]...,"False. In a multitasking case, the same progra...",FALSE\nEach process is an independent entity a...,2,Well done! Your answer clearly explains why th...
2037,NaN,Q1. State TRUE or FALSE and justify. No correc...,False. If any system calls are made by a user ...,"False, OS can use system calls as an event to ...",2,Your answer is correct. You have effectively e...


In [15]:
train_df.isna().sum()

Question_id       709
Question            0
Student Answer     12
Correct Answer      0
output_label        0
feedback            0
dtype: int64

In [16]:
train_df.duplicated().sum()

np.int64(7)

In [17]:
train_df.loc[train_df.duplicated(keep=False)]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
214,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nTRUE,"True, pipe system call only allows for one dir...",0,your response is incorrect as no justificatio...
499,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse,"True, the user process has no control over whi...",0,incorrect because no justification is provided
589,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse,"True, the user process has no control over whi...",0,incorrect because no justification is provided
941,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nTRUE,"True, pipe system call only allows for one dir...",0,your response is incorrect as no justificatio...
1032,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, some other process can also be schedule...",0,incorrect because no justification is provided
1374,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFALSE,"False, only the scheduler can decide which pro...",0,no justification is provided
1410,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFALSE,"False, only the scheduler can decide which pro...",0,no justification is provided
1742,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, some other process can also be schedule...",0,incorrect because no justification is provided
2127,NaN,Q1. State TRUE or FALSE and justify. No correc...,NaN,"True, parent and child are entirely two differ...",0,Incorrect as no answer is provided by the stud...
2158,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, only the scheduler can decide which pro...",0,Incorrect because no justification is provided


In [18]:
train_df["Question_id"].nunique()

97

In [19]:
train_df.loc[train_df["Question_id"].isna()]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
0,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, parent and child processes are two inde...",0,Your answer is incorrect. Parent and child pro...
12,NaN,Q1. State TRUE or FALSE and justify. No correc...,"**TRUE**, the OS the the time sharing or conte...","True, the OS can use system calls as an event ...",1,Your answer captures the idea that the OS can ...
14,NaN,Q1. State TRUE or FALSE and justify. No correc...,**TRUE**,"True, parent and child are entirely two differ...",0,The justification provided lacks specific in...
16,NaN,Q1. State TRUE or FALSE and justify. No correc...,TRUE,"True, pipe system call only allows for one dir...",0,your response is incorrect as no justificatio...
17,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nTrue. Scheduling can be done in kernel mode ...,"True, the user process has no control over whi...",2,Your response is accurate. You correctly under...
...,...,...,...,...,...,...
3642,NaN,Q1. State TRUE or FALSE and justify. No correc...,"No, when the interrupt is of implicit-software...","False, some other process can also be schedule...",1,Your response identifies a case where the proc...
3643,NaN,Q1. State TRUE or FALSE and justify. No correc...,"\nTrue, it is possible that the process may no...","True, if the time slice of the process ends wh...",2,Your answer provides a precise and correct exp...
3644,NaN,Q1. State TRUE or FALSE and justify. No correc...,"\n$\rightarrow$ False, there is a chance that ...","False, some other process can also be schedule...",2,Well done! You've correctly identified that ...
3648,NaN,Q1. State TRUE or FALSE and justify. No correc...,"\nTrue, because write permissions are not ther...","True, pipe system call only allows for one dir...",2,Well done! You accurately stated that only uni...


In [20]:
train_df.duplicated(subset="Question").sum()

np.int64(3556)

In [21]:
train_df["Question"].nunique()

106

In [22]:
train_df.loc[train_df.duplicated(subset="Question")]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
5,301390.0,To segment the rose petals [4 marks]:,A narrow range around 0 for hue,To segment rose petals\nH (Hue) -- Narrow rang...,1,Your answer is partially correct. While you ...
9,301390.0,To segment the rose petals [4 marks]:,a narrow range around 0 degrees of H,To segment rose petals\nH (Hue) -- Narrow rang...,1,The answer is partially correct. Including h...
10,42544.0,a. List and explain two privileged actions tha...,The program might try to disable the timer in...,- Write to CR3 register to modify page directo...,1,The first response is correct as it identifies...
18,219127.0,Two advantages of separating declaration ...,1) shows all the function declarations of the ...,The code will be more clear and easier for the...,2,Well done! You have correctly identified the...
26,275834.0,Write three parameters which affect ...,"particle size, ultrasonic frequency , tool cro...","Frequency of vibration, Amplitude of vibration...",1,Feedback: You have identified two correct pa...
...,...,...,...,...,...,...
3657,248107.0,What is the role of an equalization tank in wa...,1. to balance the fluctuating flow of water\n2...,The equalization tanks are provided to-\n1. To...,2,Very well done! Your answer precisely aligns w...
3658,248100.0,List four factors affecting water consumption.,1) Climatic conditions b\n2) Geography of the ...,1) Climatic conditions\n2) geography of the re...,2,Excellent work! You have a clear understanding...
3659,NaN,Q1. State TRUE or FALSE and justify. No correc...,"\nFalse, the order of parent, child execution ...","False, parent and child processes are two inde...",1,Correctly stated that parent and child are ind...
3660,42544.0,a. List and explain two privileged actions tha...,Modifying kernel registers - Consider the reg...,- Write to CR3 register to modify page directo...,1,There are two important issues with your answe...


In [23]:
train_df.duplicated(subset="Student Answer").sum()

np.int64(145)

In [24]:
train_df["Student Answer"].nunique()

3516

In [25]:
train_df.loc[train_df.duplicated(subset="Student Answer")]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
34,NaN,Q1. State TRUE or FALSE and justify. No correc...,**TRUE**,"True, if the time slice of the process ends wh...",0,Try to explain the scenario better. The reason...
37,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, only the scheduler can decide which pro...",0,Justify your response. User processes do not h...
245,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse,"False, some other process can also be schedule...",0,incorrect because no justification is provided
325,324707.0,What is the difference between basin order and...,Order of Basin is the order of its highest ord...,Basin order is highest order of any stream in ...,1,Great observation on the role of channel order...
358,NaN,Q1. State TRUE or FALSE and justify. No correc...,**TRUE**,"True, if the time slice of the process ends wh...",0,no justification is provided
...,...,...,...,...,...,...
3504,NaN,Q1. State TRUE or FALSE and justify. No correc...,True.,"True, if the time slice of the process ends wh...",0,A justification for your answer is required fo...
3524,262097.0,Write any advantage of using TiC/TiN as a tool...,Excellent wear resistance,excellent wear resistance\nthermal stability\n...,2,"That's a good start. However, there are addi..."
3530,42546.0,c. When the fork() system call is made the chi...,4,- When a child process is created and is ready...,0,The provided answer does not address the que...
3569,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFALSE,"False, some other process can also be schedule...",0,The student correctly identified the stateme...


In [26]:
train_df.duplicated(subset="Correct Answer").sum()

np.int64(3555)

In [27]:
train_df["Correct Answer"].nunique()

107

In [28]:
train_df.loc[train_df.duplicated(subset="Correct Answer")]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
5,301390.0,To segment the rose petals [4 marks]:,A narrow range around 0 for hue,To segment rose petals\nH (Hue) -- Narrow rang...,1,Your answer is partially correct. While you ...
9,301390.0,To segment the rose petals [4 marks]:,a narrow range around 0 degrees of H,To segment rose petals\nH (Hue) -- Narrow rang...,1,The answer is partially correct. Including h...
10,42544.0,a. List and explain two privileged actions tha...,The program might try to disable the timer in...,- Write to CR3 register to modify page directo...,1,The first response is correct as it identifies...
18,219127.0,Two advantages of separating declaration ...,1) shows all the function declarations of the ...,The code will be more clear and easier for the...,2,Well done! You have correctly identified the...
26,275834.0,Write three parameters which affect ...,"particle size, ultrasonic frequency , tool cro...","Frequency of vibration, Amplitude of vibration...",1,Feedback: You have identified two correct pa...
...,...,...,...,...,...,...
3657,248107.0,What is the role of an equalization tank in wa...,1. to balance the fluctuating flow of water\n2...,The equalization tanks are provided to-\n1. To...,2,Very well done! Your answer precisely aligns w...
3658,248100.0,List four factors affecting water consumption.,1) Climatic conditions b\n2) Geography of the ...,1) Climatic conditions\n2) geography of the re...,2,Excellent work! You have a clear understanding...
3659,NaN,Q1. State TRUE or FALSE and justify. No correc...,"\nFalse, the order of parent, child execution ...","False, parent and child processes are two inde...",1,Correctly stated that parent and child are ind...
3660,42544.0,a. List and explain two privileged actions tha...,Modifying kernel registers - Consider the reg...,- Write to CR3 register to modify page directo...,1,There are two important issues with your answe...


In [29]:
train_df.duplicated(subset="feedback").sum()

np.int64(48)

In [30]:
train_df["feedback"].nunique()

3614

In [31]:
train_df.loc[train_df.duplicated(subset="feedback")]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
132,NaN,Q1. State TRUE or FALSE and justify. No correc...,True.,"False, OS can use system calls as an event to ...",0,incorrect because no justification is provided
214,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nTRUE,"True, pipe system call only allows for one dir...",0,your response is incorrect as no justificatio...
245,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse,"False, some other process can also be schedule...",0,incorrect because no justification is provided
499,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse,"True, the user process has no control over whi...",0,incorrect because no justification is provided
584,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFALSE,"False, OS can use system calls as an event to ...",0,incorrect because no justification is provided
589,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse,"True, the user process has no control over whi...",0,incorrect because no justification is provided
628,NaN,Q1. State TRUE or FALSE and justify. No correc...,TRUE,"False, write can only be done on fd[1] and rea...",0,your response is incorrect as no justificatio...
787,NaN,Q1. State TRUE or FALSE and justify. No correc...,\n**False**,"False, write can only be done on fd[1] and rea...",0,your response is incorrect as no justificatio...
862,NaN,Q1. State TRUE or FALSE and justify. No correc...,**True**,"True, if the time slice of the process ends wh...",0,no justification is provided
941,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nTRUE,"True, pipe system call only allows for one dir...",0,your response is incorrect as no justificatio...


In [32]:
train_df["output_label"].value_counts()

output_label
2    1548
1    1268
0     846
Name: count, dtype: int64

In [33]:
train_df["output_label"].value_counts(normalize=True)

output_label
2    0.422720
1    0.346259
0    0.231021
Name: proportion, dtype: float64

In [34]:
train_df.drop(columns=["Question_id"], inplace=True)

In [35]:
train_df.dropna(inplace=True)

In [36]:
train_df.drop_duplicates(inplace=True)

In [37]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3645 entries, 0 to 3661
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        3645 non-null   object
 1   Student Answer  3645 non-null   object
 2   Correct Answer  3645 non-null   object
 3   output_label    3645 non-null   int64 
 4   feedback        3645 non-null   object
dtypes: int64(1), object(4)
memory usage: 170.9+ KB


In [38]:
train_df["output_label"].value_counts()

output_label
2    1548
1    1268
0     829
Name: count, dtype: int64

In [39]:
train_df["output_label"].value_counts(normalize=True)

output_label
2    0.424691
1    0.347874
0    0.227435
Name: proportion, dtype: float64

In [40]:
train_df.loc[train_df.duplicated(subset=["Question", "Student Answer"])]

,Question,Student Answer,Correct Answer,output_label,feedback
325,What is the difference between basin order and...,Order of Basin is the order of its highest ord...,Basin order is highest order of any stream in ...,1,Great observation on the role of channel order...
358,Q1. State TRUE or FALSE and justify. No correc...,**TRUE**,"True, if the time slice of the process ends wh...",0,no justification is provided
554,c. When the fork() system call is made the chi...,4,- When a child process is created and is ready...,0,Your answer is incorrect. The child process do...
568,Two methods of estimating parameters of the pr...,"method of moments, method of maximum likelihood",Method of moments and Method of maximum likeli...,2,That's right! You've correctly identified the ...
627,\nWhat\nis the Full form of RCP?,Representative Concentration Pathway,Representative Concentration Pathway,2,Great! You have provided the accurate full ...
...,...,...,...,...,...
3504,Q1. State TRUE or FALSE and justify. No correc...,True.,"True, if the time slice of the process ends wh...",0,A justification for your answer is required fo...
3524,Write any advantage of using TiC/TiN as a tool...,Excellent wear resistance,excellent wear resistance\nthermal stability\n...,2,"That's a good start. However, there are addi..."
3530,c. When the fork() system call is made the chi...,4,- When a child process is created and is ready...,0,The provided answer does not address the que...
3569,Q1. State TRUE or FALSE and justify. No correc...,\nFALSE,"False, some other process can also be schedule...",0,The student correctly identified the stateme...


In [41]:
train_df.loc[train_df.duplicated(subset=["Question", "Student Answer", "Correct Answer"])]

,Question,Student Answer,Correct Answer,output_label,feedback
325,What is the difference between basin order and...,Order of Basin is the order of its highest ord...,Basin order is highest order of any stream in ...,1,Great observation on the role of channel order...
358,Q1. State TRUE or FALSE and justify. No correc...,**TRUE**,"True, if the time slice of the process ends wh...",0,no justification is provided
554,c. When the fork() system call is made the chi...,4,- When a child process is created and is ready...,0,Your answer is incorrect. The child process do...
568,Two methods of estimating parameters of the pr...,"method of moments, method of maximum likelihood",Method of moments and Method of maximum likeli...,2,That's right! You've correctly identified the ...
627,\nWhat\nis the Full form of RCP?,Representative Concentration Pathway,Representative Concentration Pathway,2,Great! You have provided the accurate full ...
...,...,...,...,...,...
3504,Q1. State TRUE or FALSE and justify. No correc...,True.,"True, if the time slice of the process ends wh...",0,A justification for your answer is required fo...
3524,Write any advantage of using TiC/TiN as a tool...,Excellent wear resistance,excellent wear resistance\nthermal stability\n...,2,"That's a good start. However, there are addi..."
3530,c. When the fork() system call is made the chi...,4,- When a child process is created and is ready...,0,The provided answer does not address the que...
3569,Q1. State TRUE or FALSE and justify. No correc...,\nFALSE,"False, some other process can also be schedule...",0,The student correctly identified the stateme...


In [42]:
train_df.loc[train_df.duplicated(subset=["Question", "Student Answer", "feedback"])]

,Question,Student Answer,Correct Answer,output_label,feedback


## Test Dataset (`unseen_answers`)
Here is a summary of the insights and finds of the unseen_answers set:

1. Same as points 1, 2, 3, 4, 5, 6, 11, 12, and 13 in the raw train dataset.
2. the total size is 980 records.
3. there is 103 unique questions.
4. there is 954 unique student answer.
5. there is 963 unique feedback.

In [4]:
test_answers_path = os.path.join(raw_datasets, "unseen_answers.csv")

test_answers_df   = pd.read_csv(test_answers_path)

test_answers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 980 entries, 0 to 979
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Question_id     800 non-null    float64
 1   Question        980 non-null    object 
 2   Student Answer  978 non-null    object 
 3   Correct Answer  980 non-null    object 
 4   output_label    980 non-null    int64  
 5   feedback        980 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 46.1+ KB


In [44]:
(test_answers_df.sample(5))

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
256,301392.0,To segment the green leaves and stem [4 marks]:,Broad range of H around 120 degrees and broad ...,Hue should be centered around 120 degrees for ...,2,Excellent! Your answer effectively captures th...
414,346467.0,Why to test the level of significance in hydro...,Level of significance is defined as the fixed ...,The test of level of significance is used to t...,1,Your response correctly defines the level of...
194,219127.0,Two advantages of separating declaration ...,To avoid problems when our class/function /wha...,The code will be more clear and easier for the...,0,The answer is incorrect. It fails to capture...
313,262097.0,Write any advantage of using TiC/TiN as a tool...,"increases hardness, decreases wear, increases ...",excellent wear resistance\nthermal stability\n...,2,Your answer is correct. You have identified ...
376,42541.0,Q1. State TRUE or FALSE and justify. [6 marks]...,"False, as interruptable can be used to instant...",FALSE\nEach process is an independent entity a...,2,That's right! A shared PCB process-control b...


In [45]:
test_answers_df.isna().sum()

Question_id       180
Question            0
Student Answer      2
Correct Answer      0
output_label        0
feedback            0
dtype: int64

In [46]:
test_answers_df.duplicated().sum()

np.int64(2)

In [47]:
test_answers_df.loc[test_answers_df.duplicated(keep=False)]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
174,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, some other process can also be schedule...",0,incorrect because no justification is provided
178,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, some other process can also be schedule...",0,incorrect because no justification is provided
847,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"False, some other process can also be schedule...",0,incorrect because no justification is provided


In [48]:
test_answers_df["Question_id"].nunique()

94

In [49]:
test_answers_df.loc[test_answers_df["Question_id"].isna()]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
6,NaN,Q1. State TRUE or FALSE and justify. No correc...,"False, there is no such hindrance from the sta...","False, parent and child processes are two inde...",2,Your answer acknowledges the independence of p...
8,NaN,Q1. State TRUE or FALSE and justify. No correc...,opportunities to schedule READY processes and ...,"True, the OS can use system calls as an event ...",2,Well done! You have successfully identified th...
12,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE. It takes some time to restore prev. reg...,"False, some other process can also be schedule...",2,You have correctly identified that the process...
13,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nTrue. If all processes were built such that ...,"True, the OS can use system calls as an event ...",2,Your answer demonstrates a clear understanding...
22,NaN,Q1. State TRUE or FALSE and justify. No correc...,FALSE,"True, the OS can use system calls as an event ...",0,incorrect because no justification is provided
...,...,...,...,...,...,...
940,NaN,Q1. State TRUE or FALSE and justify. No correc...,\n **FALSE**,"False, only the scheduler can decide which pro...",0,Justification: The decision to schedule a proc...
954,NaN,Q1. State TRUE or FALSE and justify. No correc...,"\nTrue: In a co-operative environment, the pro...","True, the OS can use system calls as an event ...",2,Your answer showcases a good understanding of ...
955,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse. Scheduling is a privileged operation ...,"False, only the scheduler can decide which pro...",2,Your answer accurately points out that schedul...
965,NaN,Q1. State TRUE or FALSE and justify. No correc...,Ans: FALSE -> OS time sharing program in CPU.,"False, OS can use system calls as an event to ...",1,Your answer points out a fundamental role of t...


In [50]:
test_answers_df.duplicated(subset="Question").sum()

np.int64(877)

In [51]:
test_answers_df["Question"].nunique()

103

In [53]:
test_answers_df.loc[test_answers_df.duplicated(subset="Question")]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
4,301390.0,To segment the rose petals [4 marks]:,HSI can decouple the color information and int...,To segment rose petals\nH (Hue) -- Narrow rang...,1,Your answer displays a good understanding of...
5,301390.0,To segment the rose petals [4 marks]:,A narrow range of I centered around 0.33 as R ...,To segment rose petals\nH (Hue) -- Narrow rang...,1,Your answer correctly identifies the narrow ra...
13,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nTrue. If all processes were built such that ...,"True, the OS can use system calls as an event ...",2,Your answer demonstrates a clear understanding...
15,219127.0,Two advantages of separating declaration ...,1.we can separate interface and implementation...,The code will be more clear and easier for the...,1,1. The response could have addressed more poin...
18,219127.0,Two advantages of separating declaration ...,1) This way we can use other structures or cla...,The code will be more clear and easier for the...,2,Your answer indicates a good understanding of ...
...,...,...,...,...,...,...
975,148021.0,Define Priority Inversion in single line.,Using API provided in freertos we can change p...,a higher priority process not getting CPU due ...,0,Your response addresses a way to change task...
976,324832.0,What are the issues that arise when number of ...,Too many hidden layer takes too long to train ...,when the number of hidden layer is very large ...,2,The response captures the key idea of overfitt...
977,324708.0,Why the steeper main streams results in the Hy...,The steeper main streams due to their higher s...,Steeper basin streams allows the run off to l...,1,While your response correctly mentions the fas...
978,42541.0,Q1. State TRUE or FALSE and justify. [6 marks]...,"False, even though multiple processes can be e...",FALSE\nEach process is an independent entity a...,0,Your answer is not entirely correct. While it'...


In [5]:
test_answers_df["Student Answer"].nunique()

954

In [6]:
test_answers_df["feedback"].nunique()

963

In [54]:
test_answers_df["output_label"].value_counts()

output_label
2    403
1    344
0    233
Name: count, dtype: int64

In [55]:
test_answers_df["output_label"].value_counts(normalize=True)

output_label
2    0.411224
1    0.351020
0    0.237755
Name: proportion, dtype: float64

In [56]:
test_answers_df.drop(columns=["Question_id"], inplace=True)

In [57]:
test_answers_df.dropna(inplace=True)

In [58]:
test_answers_df.drop_duplicates(inplace=True)

In [59]:
test_answers_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 976 entries, 0 to 979
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        976 non-null    object
 1   Student Answer  976 non-null    object
 2   Correct Answer  976 non-null    object
 3   output_label    976 non-null    int64 
 4   feedback        976 non-null    object
dtypes: int64(1), object(4)
memory usage: 45.8+ KB


In [60]:
test_answers_df["output_label"].value_counts()

output_label
2    403
1    344
0    229
Name: count, dtype: int64

In [61]:
test_answers_df["output_label"].value_counts(normalize=True)

output_label
2    0.412910
1    0.352459
0    0.234631
Name: proportion, dtype: float64

In [62]:
test_answers_df.duplicated(subset=["Question", "Student Answer", "feedback"]).sum()

np.int64(0)

## Test Dataset (`unseen_question`)

Here is a summary of the insights and finds of the unseen_question set:

1. there is no missing values.
2. there is no fully duplications.
3. the total size is 765 records.
4. there is 12 unique questions.
5. there is 751 unique student answer.
6. there is 765 unique feedback.

In [7]:
test_question_path = os.path.join(raw_datasets, "unseen_question.csv")

test_question_df   = pd.read_csv(test_question_path)

test_question_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Question_id     765 non-null    float64
 1   Question        765 non-null    object 
 2   Student Answer  765 non-null    object 
 3   Correct Answer  765 non-null    object 
 4   output_label    765 non-null    int64  
 5   feedback        765 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 36.0+ KB


In [64]:
test_question_df.sample(5)

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
511,24.0,Q1. State TRUE or FALSE and justify. [6 marks]...,False. CPU provides OS with ISA which builds u...,FALSE\nThe ISA is an abstraction ins...,1,"Your justification is mostly correct. However,..."
192,260815.0,What are the factors that affect the evaporati...,"Storage,Wind,Water Quality, Area, Heat sored, ...","Wind Speed, Temperature of water and air, vapo...",2,Well done! Your answer encompasses the key fac...
441,24.0,Q1. State TRUE or FALSE and justify. [6 marks]...,False - The OS uses the instruction set archit...,FALSE\nThe ISA is an abstraction ins...,2,Well done! You correctly explained that the OS...
201,260815.0,What are the factors that affect the evaporati...,"Surface area, water temperature (air temperatu...","Wind Speed, Temperature of water and air, vapo...",2,Very well explained! Your answer covers all th...
698,23.0,Q1. State TRUE or FALSE and justify. [6 marks]...,False - It is not necessary that the process ...,FALSE\nIf the lock is available a spinlock doe...,2,Well done! You provided a well-reasoned justif...


In [65]:
test_question_df.duplicated().sum()

np.int64(0)

In [66]:
test_question_df["Question_id"].nunique()

12

In [67]:
test_question_df.loc[test_question_df.duplicated(subset="Question")]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
1,346456.0,What are the key assumptions of Muskingum floo...,This method assumes a single stage- discharge ...,The Muskingum assumes singles stage. It has th...,1,Your understanding of the singular stage-disch...
2,346456.0,What are the key assumptions of Muskingum floo...,This method assumes a single stage- discharge ...,The Muskingum assumes singles stage. It has th...,1,Your answer highlights one of the key assumpti...
3,346456.0,What are the key assumptions of Muskingum floo...,This method assumes a single stage- discharge ...,The Muskingum assumes singles stage. It has th...,1,Feedback: \n\nYour response correctly iden...
4,346456.0,What are the key assumptions of Muskingum floo...,The Muskingum flood routing method gives the s...,The Muskingum assumes singles stage. It has th...,2,Well done! You have demonstrated a clear under...
5,346456.0,What are the key assumptions of Muskingum floo...,"Assumptions: For every dischange, we take one ...",The Muskingum assumes singles stage. It has th...,1,Feedback: Your understanding of the Muskingu...
...,...,...,...,...,...,...
760,23.0,Q1. State TRUE or FALSE and justify. [6 marks]...,True thats how spinlocks are implemented. They...,FALSE\nIf the lock is available a spinlock doe...,2,Your response acknowledges that a spinlock doe...
761,23.0,Q1. State TRUE or FALSE and justify. [6 marks]...,False. Thread/process will only spin on CPU i...,FALSE\nIf the lock is available a spinlock doe...,2,Your answer is accurate. You correctly stated ...
762,23.0,Q1. State TRUE or FALSE and justify. [6 marks]...,"True, if the lock isn't available, the process...",FALSE\nIf the lock is available a spinlock doe...,1,"Your justification is correct. However, a spin..."
763,23.0,Q1. State TRUE or FALSE and justify. [6 marks]...,True: This is the definition of spinlock. The ...,FALSE\nIf the lock is available a spinlock doe...,2,Your answer demonstrates a clear understanding...


In [68]:
test_question_df.loc[test_question_df.duplicated(subset=["Question", "Student Answer"])]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
1,346456.0,What are the key assumptions of Muskingum floo...,This method assumes a single stage- discharge ...,The Muskingum assumes singles stage. It has th...,1,Your understanding of the singular stage-disch...
2,346456.0,What are the key assumptions of Muskingum floo...,This method assumes a single stage- discharge ...,The Muskingum assumes singles stage. It has th...,1,Your answer highlights one of the key assumpti...
20,344833.0,What is the role of edge direction in Harris C...,Edge direction plays an important role as it h...,The role of edge direction in Harris Corner de...,1,Your answer partially addresses the role of ...
53,344833.0,What is the role of edge direction in Harris C...,The role of edge direction in Harris Corner de...,The role of edge direction in Harris Corner de...,2,Excellent! You have a clear understanding of t...
266,260785.0,Expand the term SWOT,surface water and ocean topography,SWOT stands for Strengh Weakness Opportunity T...,0,"""SWOT"" stands for specific business-related fa..."
272,260785.0,Expand the term SWOT,Strength Weakness Opportunity Threat,SWOT stands for Strengh Weakness Opportunity T...,2,Your answer is correct. You have accurately ...
273,260785.0,Expand the term SWOT,strength weakness opportunity threat,SWOT stands for Strengh Weakness Opportunity T...,2,Well done! You have correctly defined the acr...
276,260785.0,Expand the term SWOT,strength weakness opportunity threat,SWOT stands for Strengh Weakness Opportunity T...,2,Well done! You have correctly expanded the SWO...
287,260785.0,Expand the term SWOT,surface water and ocean topography,SWOT stands for Strengh Weakness Opportunity T...,0,The provided answer does not relate to SWOT an...
289,260785.0,Expand the term SWOT,Strength Weakness Opportunity Threat,SWOT stands for Strengh Weakness Opportunity T...,2,Great job! You've correctly expanded the term ...


In [69]:
test_question_df.loc[test_question_df.duplicated(subset=["Question", "Student Answer", "feedback"])]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback


In [8]:
test_question_df["Question"].nunique()

12

In [9]:
test_question_df["Student Answer"].nunique()

751

In [10]:
test_question_df["feedback"].nunique()

765

In [70]:
test_question_df["output_label"].value_counts()

output_label
2    321
1    278
0    166
Name: count, dtype: int64

In [71]:
test_question_df["output_label"].value_counts(normalize=True)

output_label
2    0.419608
1    0.363399
0    0.216993
Name: proportion, dtype: float64

In [72]:
test_question_df.drop(columns=["Question_id"], inplace=True)

## Validation Dataset
Here is a summary of the insights and finds of the val set:

1. there is missing values.
2. the is no fully duplications.
3. the total size is 407 records.
4. there is 90 unique questions.
5. there is 400 unique student answer.
6. there is 403 unique feedback.

In [11]:
validation_path = os.path.join(raw_datasets, "val.csv")

validation_df   = pd.read_csv(validation_path)

validation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Question_id     316 non-null    float64
 1   Question        407 non-null    object 
 2   Student Answer  405 non-null    object 
 3   Correct Answer  407 non-null    object 
 4   output_label    407 non-null    int64  
 5   feedback        407 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 19.2+ KB


In [74]:
validation_df.sample(5)

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
92,42541.0,Q1. State TRUE or FALSE and justify. [6 marks]...,"False, each process has its own separate PLB. ...",FALSE\nEach process is an independent entity a...,2,Well done! Your understanding of the reasons b...
206,46882.0,During coagulation process alkalinity reduces ...,anions like co3(2-) get precipitated with meta...,"As coagulation is done,the OH- tend to form mo...",0,Consider looking into the chemical reactions o...
279,226518.0,Describe the purpose of state DecideAction in ...,to decide the next action of the robot when it...,DecideAction node was used to decide which dir...,1,Your response captures the essence of the node...
233,219127.0,Two advantages of separating declaration ...,1 . we can easily figure out which member func...,The code will be more clear and easier for the...,0,The provided answer does not cover key point...
253,131601.0,Define meteorological drought.,happens when dry weather pattern dominantes ov...,meteorological draught is caused by the condit...,1,Your response correctly identifies a key fac...


In [75]:
validation_df.isna().sum()

Question_id       91
Question           0
Student Answer     2
Correct Answer     0
output_label       0
feedback           0
dtype: int64

In [76]:
validation_df.duplicated().sum()

np.int64(0)

In [77]:
validation_df["Question_id"].nunique()

81

In [78]:
validation_df.loc[validation_df["Question_id"].isna()]

,Question_id,Question,Student Answer,Correct Answer,output_label,feedback
3,NaN,Q1. State TRUE or FALSE and justify. No correc...,True.,"True, parent and child are entirely two differ...",0,"Consider the fact that after forking, the chil..."
4,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse. The OS can still schedule READY proce...,"False, OS can use system calls as an event to ...",2,Well done for identifying that the operating s...
8,NaN,Q1. State TRUE or FALSE and justify. No correc...,"False**. after the fork(), both the parent an...","True, parent and child are entirely two differ...",0,The write operation in exec doesn't depend o...
27,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse. OS decides whether to resume it or sw...,"False, some other process can also be schedule...",2,Your response correctly identifies that choosi...
33,NaN,Q1. State TRUE or FALSE and justify. No correc...,"FALSE, parent process has the ability to initi...","False, only the scheduler can decide which pro...",2,Your understanding of the capabilities of a pa...
...,...,...,...,...,...,...
380,NaN,Q1. State TRUE or FALSE and justify. No correc...,True The second descriptor is write only and t...,"True, pipe system call only allows for one dir...",2,Excellent understanding of the pipe system cal...
388,NaN,Q1. State TRUE or FALSE and justify. No correc...,False.,"False, OS can use system calls as an event to ...",0,incorrect because no justification is provided
391,NaN,Q1. State TRUE or FALSE and justify. No correc...,\nFalse. The scheduling of process is only han...,"False, only the scheduler can decide which pro...",2,Your answer is correct. You accurately stated ...
396,NaN,Q1. State TRUE or FALSE and justify. No correc...,False. It may be rescheduled.,"True, if the time slice of the process ends wh...",0,You are correct that the process may be resche...


In [79]:
validation_df.duplicated(subset="Question").sum()

np.int64(317)

In [12]:
validation_df["Question"].nunique()

90

In [80]:
validation_df["Student Answer"].nunique()

400

In [81]:
validation_df["feedback"].nunique()

403

In [82]:
validation_df["output_label"].value_counts()

output_label
2    168
1    144
0     95
Name: count, dtype: int64

In [83]:
validation_df["output_label"].value_counts(normalize=True)

output_label
2    0.412776
1    0.353808
0    0.233415
Name: proportion, dtype: float64

In [84]:
validation_df.drop(columns=["Question_id"], inplace=True)

In [85]:
validation_df.dropna(inplace=True)

In [86]:
validation_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 405 entries, 0 to 406
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        405 non-null    object
 1   Student Answer  405 non-null    object
 2   Correct Answer  405 non-null    object
 3   output_label    405 non-null    int64 
 4   feedback        405 non-null    object
dtypes: int64(1), object(4)
memory usage: 19.0+ KB


In [87]:
validation_df["output_label"].value_counts()

output_label
2    168
1    144
0     93
Name: count, dtype: int64

In [88]:
validation_df["output_label"].value_counts(normalize=True)

output_label
2    0.414815
1    0.355556
0    0.229630
Name: proportion, dtype: float64

In [89]:
validation_df.duplicated(subset=["Question", "Student Answer", "feedback"]).sum()

np.int64(0)

## Semantic Similarities (Leakages)
Here our goal is to answer these questions: How many leakages is there? What we'll do with them?

Here is the summary about what we did and what we found:
1. convert the dataframes into list of texts, each text represents the concatenated columns value.
2. we used the FAISS (facebook AI similarity search) to see how many semantic similarities between the train set and the others.
3. the threshold is 90%, where filter the records that have similarity score 90% or above.
4. the threshold value wasn’t chosen arbitrarily. we started testing at 80% and increased it gradually. At 80%, too many records were flagged due to repeated `Question` and `Correct Answer`, making the similarity score too high. To reduce false positives and better capture genuinely similar `Student Answer` and `feedback`, we raised the threshold to 90%, which proved to be the most suitable.
5. we used `all-MiniLM-L6-v2` sentence-transformers for embeddings.
6. we used `all-MiniLM-L6-v2` model because it is small, fast and very good for short sentences (~512 tokens).
7. after cleaning the datasets from missing values and direct duplications; here is the total size, leakages, non-leakages for each:
   1. `unseen_answers`: **total size**: 976, **leakages with training set**: 909, **non-leakages**: 67
   2. `unseen_question`: **total size**: 765, **leakages with training set**: 17, **non-leakages**: 748
   3. `val`: **total size**: 405, **leakages with training set**: 376, **non-leakages**: 29
8. the leakage indices are saved in `leakage_indices.json` to use them in the preprocessing stage.

In [90]:
def to_text(df):
    """
    Convert all rows of a DataFrame to concatenated text strings.

    :param df: Pandas DataFrame
    :return: Concatenated text strings
    """
    return df[df.columns.tolist()].astype(str).agg(" ".join, axis=1).tolist()


train_texts         = to_text(train_df)
test_answers_texts  = to_text(test_answers_df)
test_question_texts = to_text(test_question_df)
validation_texts    = to_text(validation_df)

In [91]:
def detect_leakage(query_texts, query_name, index, threshold=0.9, with_info=False):
    """
    Detect potential data leakages by finding query entries (e.g., unseen answers) too similar to training data.

    :param query_texts: list of text entries to check for leakage.
    :param query_name: label used for printing and identification.
    :param index: FAISS index built on normalized training embeddings.
    :param threshold: similarity threshold above which leakage is flagged.
    :return: list of indices in query_texts that are considered leakages.
    """
    query_embeddings = model.encode(query_texts)

    faiss.normalize_L2(query_embeddings)

    D, I             = index.search(query_embeddings, k=1)

    leakage_indices  = [i for i, score in enumerate(D[:, 0]) if score >= threshold]

    if with_info:
        for i in leakage_indices:
            print(f"{query_name}[{i}]: {query_texts[i]}")

            train_match_idx = int(I[i][0])

            print(f"Matched Train[{train_match_idx}]: {train_texts[train_match_idx]}")

            print(f"Similarity: {D[i][0]:.4f}")

            print("=" * 80)

        print(f"Number of Leakages in {query_name}: {len(leakage_indices)}\n")

        return None

    return leakage_indices

In [92]:
model            = SentenceTransformer("all-MiniLM-L6-v2")

train_embeddings = model.encode(train_texts)
faiss.normalize_L2(train_embeddings)
index            = faiss.IndexFlatIP(train_embeddings.shape[1])

index.add(train_embeddings)

### `unseen_answers` & `train`

In [93]:
detect_leakage(query_texts=test_answers_texts, query_name="test_answers_texts", index=index, with_info=True)

test_answers_texts[1]: How is regularized set operation helpful in solid modeling? It helps in joining and maintaining local curvature of two objects. boundaries are defined. no dangling edges. no edges are created extra and no edges are lost while taking intersection of two objects.Regularized set operation helps to follow the defined process in predefined sequence 0   The answer lacks the essential information that regularized set operation in solid modeling helps to define boundaries, eliminate dangling edges, and maintain object integrity during operations like intersection.   
Matched Train[1926]: How is regularized set operation helpful in solid modeling? boundaries are defined. no dangling edges. no edges are created extra and no edges are lost while taking intersection of two objects boundaries are defined. no dangling edges. no edges are created extra and no edges are lost while taking intersection of two objects.Regularized set operation helps to follow the defined process in

### `unseen_question` & `train`

In [94]:
detect_leakage(query_texts=test_question_texts, query_name="test_question_texts", index=index, with_info=True)

test_question_texts[348]: During coagulation process alkalinity reduces due to the ___________________________ Formation of metal hydroxide As coagulation is done,the OH- tend to form more bonds with the molecules in the sample, and H+ concentration increases which lowers the pH level. 0   The provided response does not address the reduction in alkalinity during the coagulation process. It only mentions the formation of metal hydroxide without explaining the connection to alkalinity reduction.   
Matched Train[1754]: During coagulation process alkalinity reduces due to the ___________________________ due to consumption of OH-. As coagulation is done,the OH- tend to form more bonds with the molecules in the sample, and H+ concentration increases which lowers the pH level. 1   You are on the right track. The alkalinity reduction is due to the consumption of hydroxide ions, but it is also important to recognize that this is a result of the formation of metal hydroxide precipitates, which 

### `val` & `train`

In [95]:
detect_leakage(query_texts=validation_texts, query_name="validation_texts", index=index, with_info=True)

validation_texts[0]: Write three      parameters which      affect MRR in ultrasonic machining ? particle size, tool workpiece gap, ultrasonic frequency Frequency of vibration, Amplitude of vibration, Abrasive slurry concentration, Slurry Particle Diameter 1 Feedback:   The answer includes some correct parameters like ultrasonic frequency and particle size. However, tool workpiece gap is not a parameter that affects MRR in ultrasonic machining. Consider including parameters like amplitude of vibration and abrasive slurry concentration to improve the comprehensiveness of the answer.   
Matched Train[2938]: Write three      parameters which      affect MRR in ultrasonic machining ? workpiece material, slurry concentration, tool material, static load Frequency of vibration, Amplitude of vibration, Abrasive slurry concentration, Slurry Particle Diameter 1   You have mentioned the parameters that are important for ultrasonic machining but not specifically for MRR. Please check the question 

In [96]:
unseen_answers_leakages     = detect_leakage(query_texts=test_answers_texts, query_name="test_answers_texts", index=index)

unseen_answers_non_leakages = len(test_answers_texts) - len(unseen_answers_leakages)

print("Total Size        : ", len(test_answers_texts))
print("Number of Leakages: ", len(unseen_answers_leakages))
print("Non Leakages      : ", unseen_answers_non_leakages)

Total Size        :  976
Number of Leakages:  909
Non Leakages      :  67


In [97]:
unseen_question_leakages     = detect_leakage(query_texts=test_question_texts, query_name="test_question_texts", index=index)

unseen_question_non_leakages = len(test_question_texts) - len(unseen_question_leakages)

print("Total Size        : ", len(test_question_texts))
print("Number of Leakages: ", len(unseen_question_leakages))
print("Non Leakages      : ", unseen_question_non_leakages)

Total Size        :  765
Number of Leakages:  17
Non Leakages      :  748


In [98]:
validation_leakages     = detect_leakage(query_texts=validation_texts , query_name="validation_texts" , index=index)

validation_non_leakages = len(validation_texts) - len(validation_leakages)

print("Total Size        : ", len(validation_texts))
print("Number of Leakages: ", len(validation_leakages))
print("Non Leakages      : ", validation_non_leakages)

Total Size        :  405
Number of Leakages:  376
Non Leakages      :  29


In [99]:
leakage_indices = {"unseen_answers" : unseen_answers_leakages,
                   "unseen_question": unseen_question_leakages,
                   "val"            : validation_leakages}

with open("leakage_indices.json", "w") as file: json.dump(leakage_indices, file, indent=4)

## Inspect Leakages & Non-Leakages
Howe we will take the advantage of leakages and non-leakages?

1. the main reason high similarities that because of the `Question`, `Correct Answer` and `output_label` are duplicated and this is normal, why? because the essential variables are the `Student Answer` and `feedback`, as each student responds differently to the same question. For example, in an exam with 20 students, one question appears 20 times with 20 unique answers and 20 corresponding `feedback`.
2. we will isolate the non-leakage records to use them for validation and test.
3. the split with the most leakages is the `unseen_answers` split, while the split with the fewest leakages is the `unseen_question` split.
4. given the ratio of split size to the number of leakages, the `val` split is unsuitable for direct validation, as it is nearly fully leaky, with only 29 non-leakages records. so, we need to resplit the test and validation data.

 ### `unseen_answers`

In [100]:
test_answers_leakages_df     = test_answers_df.iloc[unseen_answers_leakages]
test_answers_non_leakages_df = test_answers_df.drop(index=test_answers_df.index[unseen_answers_leakages])

In [101]:
test_answers_leakages_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 909 entries, 1 to 979
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        909 non-null    object
 1   Student Answer  909 non-null    object
 2   Correct Answer  909 non-null    object
 3   output_label    909 non-null    int64 
 4   feedback        909 non-null    object
dtypes: int64(1), object(4)
memory usage: 42.6+ KB


In [102]:
test_answers_leakages_df.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
57,Q1. State TRUE or FALSE and justify. [6 marks]...,False. When an executable is used to instantia...,FALSE\nEach process is an independent entity a...,2,Your answer correctly states that each process...
117,Cleaning of slow sand filter is done by ...,1.scraping of surface layer of sand and washin...,Cleaning is usually done by scraping the mediu...,1,Response recognizes two common cleaning meth...
810,To segment the green leaves and stem [4 marks]:,a narrow range around 120 degress for H,Hue should be centered around 120 degrees for ...,1,While the range around 120 degrees is approp...
947,a. List and explain two privileged actions tha...,* Accessing memory space.\nOS won't allow use...,- Write to CR3 register to modify page directo...,1,The student has identified access to memory ...
727,What is conveyed by the very low eigenvalues w...,Very low eigen value in PCT imply that those d...,A very low eigen value refers to axes (eigenve...,1,Good job noticing that low eigenvalues can i...


In [103]:
test_answers_non_leakages_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67 entries, 0 to 975
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        67 non-null     object
 1   Student Answer  67 non-null     object
 2   Correct Answer  67 non-null     object
 3   output_label    67 non-null     int64 
 4   feedback        67 non-null     object
dtypes: int64(1), object(4)
memory usage: 3.1+ KB


In [104]:
test_answers_non_leakages_df.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
496,Two methods of estimating parameters of the pr...,normal distribution and standard distribution,Method of moments and Method of maximum likeli...,0,The response does not address methods for esti...
375,Two advantages of separating declaration ...,Runtime Polymorphism- The same function name c...,The code will be more clear and easier for the...,1,While your answer mentions runtime polymorph...
517,"If there is one explanatory variable, regressi...",linear conditional regression and mul,Univariate and Multivariate,0,Make sure to provide both terms. The first t...
416,Write any two limitations of AJM.,"Low material removal rate, not good for operator","Low Material removal rate (MRR) , Limited nozz...",1,Your answer correctly identifies a limitatio...
53,Two methods of estimating parameters of the pr...,probability paper method and chi square test k...,Method of moments and Method of maximum likeli...,0,It appears that you have mentioned methods use...


In [105]:
test_answers_non_leakages_df.duplicated(subset="Student Answer").sum()

np.int64(0)

In [106]:
test_answers_non_leakages_df.duplicated(subset="feedback").sum()

np.int64(0)

 ### `unseen_question`

In [107]:
test_question_leakages_df     = test_question_df.iloc[unseen_question_leakages]
test_question_non_leakages_df = test_question_df.drop(index=test_question_df.index[unseen_question_leakages])

In [108]:
test_question_leakages_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17 entries, 348 to 371
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        17 non-null     object
 1   Student Answer  17 non-null     object
 2   Correct Answer  17 non-null     object
 3   output_label    17 non-null     int64 
 4   feedback        17 non-null     object
dtypes: int64(1), object(4)
memory usage: 816.0+ bytes


In [109]:
test_question_leakages_df.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
358,During coagulation process alkalinity reduces ...,addition of Metal ions ( e.g Alum) which act a...,"As coagulation is done,the OH- tend to form mo...",2,Your response demonstrates a solid understandi...
349,During coagulation process alkalinity reduces ...,Neutralisation of OH- or other alkaline ions,"As coagulation is done,the OH- tend to form mo...",1,The response is partially correct. While it ...
360,During coagulation process alkalinity reduces ...,addition of Fe3+ and Al3+,"As coagulation is done,the OH- tend to form mo...",0,The student's answer provides incorrect info...
365,During coagulation process alkalinity reduces ...,maintainance of pH higher and reduce NOM normal,"As coagulation is done,the OH- tend to form mo...",0,The provided answer does not address the cha...
357,During coagulation process alkalinity reduces ...,release of H+ by coagulating agent on hydrolys...,"As coagulation is done,the OH- tend to form mo...",2,Well done! You correctly understand that the...


In [110]:
test_question_non_leakages_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 748 entries, 0 to 764
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        748 non-null    object
 1   Student Answer  748 non-null    object
 2   Correct Answer  748 non-null    object
 3   output_label    748 non-null    int64 
 4   feedback        748 non-null    object
dtypes: int64(1), object(4)
memory usage: 35.1+ KB


In [111]:
test_question_non_leakages_df.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
0,What are the key assumptions of Muskingum floo...,This method assumes a single stage- discharge ...,The Muskingum assumes singles stage. It has th...,1,Your answer addresses one of the key assumptio...
186,What are the factors that affect the evaporati...,"Temperature,Relative humidity,pressure","Wind Speed, Temperature of water and air, vapo...",1,Your answer includes some important factors ...
589,Q1. State TRUE or FALSE and justify. [6 marks]...,False.\nA spinlock doesn't always check for l...,FALSE\nIf the lock is available a spinlock doe...,2,Your response accurately explains that a spinl...
237,What is vapor pressure deficit?,difference between saturate and actual vapor p...,the difference between the amount of moisture ...,2,Your answer accurately defines vapor pressure ...
82,Write any ONE limitation of the wireframe mode...,It is difficult and messy (the faces might not...,The representation is ambiguous and one wirefr...,2,Your answer acknowledges that representing a...


In [112]:
test_question_non_leakages_df.duplicated(subset="Student Answer").sum()

np.int64(14)

In [113]:
test_question_non_leakages_df.duplicated(subset="feedback").sum()

np.int64(0)

 ### `val`

In [114]:
validation_leakages_df     = validation_df.iloc[validation_leakages]
validation_non_leakages_df = validation_df.drop(index=validation_df.index[validation_leakages])

In [115]:
validation_leakages_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 376 entries, 0 to 406
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        376 non-null    object
 1   Student Answer  376 non-null    object
 2   Correct Answer  376 non-null    object
 3   output_label    376 non-null    int64 
 4   feedback        376 non-null    object
dtypes: int64(1), object(4)
memory usage: 17.6+ KB


In [116]:
validation_leakages_df.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
81,To segment the rose petals [4 marks]:,The rose petals are red (shades of red) in col...,To segment rose petals\nH (Hue) -- Narrow rang...,1,Your answer captured some aspects of the segme...
65,Q1. State TRUE or FALSE and justify. No correc...,True. The OS may schedule any other process w...,"True, if the time slice of the process ends wh...",2,Your answer captures the essence of process sc...
176,Q1. State TRUE or FALSE and justify. No correc...,\nTrue.\nThe exec system call overwrites the c...,"True, parent and child are entirely two differ...",2,Well done! You have clearly understood the con...
305,Q1. State TRUE or FALSE and justify. [6 marks]...,False - Although it is true that an incorrect ...,FALSE\nEach process is an independent entity a...,2,Excellent understanding of the topic. You corr...
118,Write three parameters which affect ...,particle size abrasive fraction frequency toll...,"Frequency of vibration, Amplitude of vibration...",1,"Your answer is partially correct, although you..."


In [117]:
validation_non_leakages_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29 entries, 28 to 392
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        29 non-null     object
 1   Student Answer  29 non-null     object
 2   Correct Answer  29 non-null     object
 3   output_label    29 non-null     int64 
 4   feedback        29 non-null     object
dtypes: int64(1), object(4)
memory usage: 1.4+ KB


In [118]:
validation_non_leakages_df.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
319,a. List and explain two privileged actions tha...,1. If user tries to access the IDT (Interrupt...,- Write to CR3 register to modify page directo...,1,"While the examples you provided are accurate, ..."
384,a. List and explain two privileged actions tha...,4. Disabling interrupts - A user process woul...,- Write to CR3 register to modify page directo...,1,Your response includes two valid privileged ...
28,a. List and explain two privileged actions tha...,Expanding process virtual address space:\nA p...,- Write to CR3 register to modify page directo...,1,You have identified possible privileged action...
385,Two advantages of separating declaration ...,1.we can separate interface and implementation...,The code will be more clear and easier for the...,1,Nicely explained! But note that having separ...
330,a. List and explain two privileged actions tha...,Some privileged actions include:\n1. When the...,- Write to CR3 register to modify page directo...,0,The answer contains partially correct informat...


In [119]:
validation_non_leakages_df.duplicated(subset="Student Answer").sum()

np.int64(0)

In [120]:
validation_non_leakages_df.duplicated(subset="feedback").sum()

np.int64(0)

## Test & Validation Splitting Strategy

What should the test and validation splits be and how?

1. concatenated all the non-leakages splits together under the name `unseen`.
2. after the concatenation, there is no fully duplications.
3. there is 14 `Student Answer` are duplicated but all the `feedback` are unique so this is what we expected and what we want.
4. the distribution of `output_label` values is almost balanced, the lowest is `0`.
5. the split size ratio will be 0.4 for the **validation** and 0.6 for the **test**, based on the concatenated `unseen` data.

In [121]:
unseen_dfs = pd.concat([test_answers_non_leakages_df, test_question_non_leakages_df,validation_non_leakages_df], ignore_index=True)

unseen_dfs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 844 entries, 0 to 843
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        844 non-null    object
 1   Student Answer  844 non-null    object
 2   Correct Answer  844 non-null    object
 3   output_label    844 non-null    int64 
 4   feedback        844 non-null    object
dtypes: int64(1), object(4)
memory usage: 33.1+ KB


In [122]:
unseen_dfs.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
770,Q1. State TRUE or FALSE and justify. [6 marks]...,False - Each attempt is not necessary to spin...,FALSE\nIf the lock is available a spinlock doe...,2,Your response is correct. You accurately state...
553,Q1. State TRUE or FALSE and justify. [6 marks]...,False ISA to hides the implementation totals o...,FALSE\nThe ISA is an abstraction ins...,2,Your answer correctly identifies that the inst...
755,Q1. State TRUE or FALSE and justify. [6 marks]...,True.\nEach time the process tries to acquire...,FALSE\nIf the lock is available a spinlock doe...,1,Your answer is partially correct. A spinlock m...
44,Give two uses of lasers,it is used in Shopping malls to scan bar code ...,"cutting and welding purpose, used in medical s...",0,The given uses are not typical applications of...
405,Name two common defluoridation techniques,co-precipitation with alum and lime(Nalgonda p...,Nalgonda Process- precipitation with alum and ...,2,Excellent! You have a solid understanding of...


In [123]:
unseen_dfs.duplicated().sum()

np.int64(0)

In [124]:
unseen_dfs.duplicated(subset="Student Answer").sum()

np.int64(14)

In [125]:
unseen_dfs.duplicated(subset="feedback").sum()

np.int64(0)

In [126]:
unseen_dfs["output_label"].value_counts()

output_label
2    337
1    319
0    188
Name: count, dtype: int64

In [127]:
unseen_dfs["output_label"].value_counts(normalize=True)

output_label
2    0.399289
1    0.377962
0    0.222749
Name: proportion, dtype: float64

### Splits Size
`validation`: 844 * 0.4 = 337.6

`test`: 844 * 0.6 = 506.4

In [128]:
validation_size = unseen_dfs.shape[0] * 0.4
test_size       = unseen_dfs.shape[0] * 0.6

validation_size, test_size

(337.6, 506.4)

## Train Splitting Strategy

What should we do with the remaining leakage records? Should we concatenate them with the training set or simply ignore them?
1. concatenated all the leakages splits together under the name `leakages`.
2. there is 3 fully duplications.
3. we dropped the fully duplications.
4. there is 35 duplicated `Student Answer`.
5. there is 18 duplicated `feedback`.
6. we concatenated the train split with `leakages`, the total size 4944.
7. after the concatenation we found:
   1. there is 3 fully duplications.
   2. we dropped the fully duplications.
   3. there is 202 duplicated `Student Answer`.
   4. there is 52 duplicated `feedback`.
   5. we dropped the duplicated `Studnet Answer`.
   6. after dropping the duplicated `Studnet Answer`, the duplicated `feedback` becomes 18.
8. the distribution of `output_label` values is almost balanced, the largest bias being for label `2` and the lowest is `0`.
9. train size before: 4944, train size after: 4735. the total number of dropped records is: 209.
10. there is 107 unique `Question`.

In [129]:
leakages_dfs = pd.concat([test_answers_leakages_df, test_question_leakages_df, validation_leakages_df], ignore_index=True)

leakages_dfs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1302 entries, 0 to 1301
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        1302 non-null   object
 1   Student Answer  1302 non-null   object
 2   Correct Answer  1302 non-null   object
 3   output_label    1302 non-null   int64 
 4   feedback        1302 non-null   object
dtypes: int64(1), object(4)
memory usage: 51.0+ KB


In [130]:
leakages_dfs.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
648,Two advantages of separating declaration ...,1. Separating declaration from definition make...,The code will be more clear and easier for the...,1,The answer is partially correct. While you men...
955,Describe the purpose of state DecideAction in ...,It is used to decide upon next corse of action...,DecideAction node was used to decide which dir...,2,Impressive! Your answer accurately captures th...
138,a. List and explain two privileged actions tha...,User process can try to disable interrupts but...,- Write to CR3 register to modify page directo...,1,While disabling interrupts is a privileged a...
730,What is the need to use homogenous coordinates...,rotation and translation can be captured in t...,"for uniform matrix representation of rotation,...",2,Your understanding of the need for homogeneous...
2,To segment the rose petals [4 marks]:,H should be around 0-10\nS should be greater t...,To segment rose petals\nH (Hue) -- Narrow rang...,1,While the ranges provided for H and S are gene...


In [131]:
leakages_dfs.duplicated().sum()

np.int64(3)

In [145]:
leakages_dfs.drop_duplicates(inplace=True)

In [146]:
leakages_dfs.duplicated(subset="Student Answer").sum()

np.int64(35)

In [147]:
leakages_dfs.duplicated(subset="feedback").sum()

np.int64(18)

In [154]:
train_with_similar = pd.concat([train_df, leakages_dfs], ignore_index=True)

train_with_similar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4944 entries, 0 to 4943
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Question        4944 non-null   object
 1   Student Answer  4944 non-null   object
 2   Correct Answer  4944 non-null   object
 3   output_label    4944 non-null   int64 
 4   feedback        4944 non-null   object
dtypes: int64(1), object(4)
memory usage: 193.2+ KB


In [155]:
train_with_similar.sample(5)

,Question,Student Answer,Correct Answer,output_label,feedback
3979,What is the role of an equalization tank in wa...,Balance fluctuating concentrations\nTo assist ...,The equalization tanks are provided to-\n1. To...,2,Your response shows a clear understanding of t...
1078,Two advantages of separating declaration ...,1) keeping all the declaration together helps ...,The code will be more clear and easier for the...,0,The provided answer does not address the adv...
730,c. When the fork() system call is made the chi...,"* After fork(), in child eip is set to forkre...",- When a child process is created and is ready...,2,Well done! You have a clear understanding of t...
911,To segment the green leaves and stem [4 marks]:,narrow range around 120 for H and narrow...,Hue should be centered around 120 degrees for ...,1,The selected values for H and I are too narrow...
3222,To segment the green leaves and stem [4 marks]:,"a narrow range around 120 in hue, a wide range...",Hue should be centered around 120 degrees for ...,2,"You correctly identified the ranges for hue,..."


In [156]:
train_with_similar.duplicated().sum()

np.int64(7)

In [157]:
train_with_similar.drop_duplicates(inplace=True)

In [158]:
train_with_similar.duplicated(subset="Student Answer").sum()

np.int64(202)

In [159]:
train_with_similar.duplicated(subset="feedback").sum()

np.int64(52)

In [160]:
train_with_similar.drop_duplicates(subset="Student Answer", inplace=True)

In [162]:
train_with_similar.duplicated(subset="feedback").sum()

np.int64(18)

In [163]:
train_with_similar["Question"].nunique()

107

In [164]:
train_with_similar["output_label"].value_counts()

output_label
2    2048
1    1674
0    1013
Name: count, dtype: int64

In [165]:
train_with_similar["output_label"].value_counts(normalize=True)

output_label
2    0.432524
1    0.353537
0    0.213939
Name: proportion, dtype: float64

### Split Size
train size: 4735 clean records.

In [166]:
train_with_similar.shape

(4735, 5)

## What is the maximum length?

To see, is there need for truncation or not? (based on the model max_length)
1. maximum length (word-based counting) in train: 481 words.
2. maximum length (word-based counting) in unseen: 361 words
3. empty prompt length (word-based counting): 34 words.

Although the text is converted to tokens instead of words, which slightly increases the length, the maximum lengths based on words are still very reasonable. Even when adding the prompt length to the longest entry in the training set, the total reaches only 515 words. Considering the models we are going to use support much larger maximum lengths, there is no need to worry about the truncation.

### `train`

In [167]:
train_with_similar_texts = to_text(train_with_similar)

max_len = max([len(text.split()) for text in train_with_similar_texts])

print("The maximum text length (words) in the train:", max_len)

The maximum text length (words) in the train: 481


### `unseen`

In [168]:
unseen_texts = to_text(unseen_dfs)

max_len = max([len(text.split()) for text in unseen_texts])

print("The maximum text length (words) in the unseen data:", max_len)

The maximum text length (words) in the unseen data: 361


### Empty Prompt

In [169]:
empty_prompt = """### Instruction:
                  Grade the student's answer based on the given mark scheme, and provide both a score and a rationale.

                  ### Input:
                  Question:
                  Reference Answer:
                  Student Answer:
                  Mark Scheme:

                  ### Response:
                  Score: , Rationale: """

print("The static length (words) of the prompt (no values are given):", len(empty_prompt.split()))

The static length (words) of the prompt (no values are given): 34
